<a href="https://colab.research.google.com/github/2303a51060Nirnaya/High_performance_computing-Hcp-/blob/main/exam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile student_mpi.c
#include <mpi.h>
#include <stdio.h>
#include <string.h>
#include <stddef.h>

struct Student {
    int id;
    char name[20];
    float marks;
};

int main(int argc, char *argv[]) {
    int rank;
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    struct Student s;

    // Define MPI datatype
    MPI_Datatype MPI_Student;
    int blocklengths[3] = {1, 20, 1};
    MPI_Aint displacements[3];
    MPI_Datatype types[3] = {MPI_INT, MPI_CHAR, MPI_FLOAT};

    displacements[0] = offsetof(struct Student, id);
    displacements[1] = offsetof(struct Student, name);
    displacements[2] = offsetof(struct Student, marks);

    MPI_Type_create_struct(3, blocklengths, displacements, types, &MPI_Student);
    MPI_Type_commit(&MPI_Student);

    if (rank == 0) {
        // Instructor process
        s.id = 101;
        strcpy(s.name, "John");
        s.marks = 89.5;

        MPI_Send(&s, 1, MPI_Student, 1, 0, MPI_COMM_WORLD);
        printf("Instructor sent student record\n");
    }
    else if (rank == 1) {
        // Student process
        MPI_Recv(&s, 1, MPI_Student, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

        printf("Received Student Data:\n");
        printf("ID: %d\n", s.id);
        printf("Name: %s\n", s.name);
        printf("Marks: %.2f\n", s.marks);
    }

    MPI_Type_free(&MPI_Student);
    MPI_Finalize();
    return 0;
}


Overwriting student_mpi.c


In [ ]:
!mpicc student_mpi.c -o student
!mpirun -np 2 --allow-run-as-root --oversubscribe ./student

Instructor sent student record
Received Student Data:
ID: 101
Name: John
Marks: 89.50
